In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import geopandas as gpd
import json

# The Data

This data was published by Patrick Ford via Kaggle at https://www.kaggle.com/datasets/patricklford/global-ev-sales-2010-2024 via the International Energy Agency, IEA, at https://www.iea.org/data-and-statistics/data-tools/global-ev-data-explorer

This goal of this notebook is to tranform the data into a format that can be visualized in a Tableau dashboard.

## Workflow Overview

This notebook loads the EV sales dataset, filters it to the most relevant historical series, reshapes it for dashboard use, and then enriches it with country geometry for mapping.

In [4]:
# Import the data
# The source file contains the global EV sales history used for the dashboard.
global_data = pd.read_csv('../Data/Global EV Data 2024.csv')

In [5]:
# Preview the first few rows to confirm the file loaded as expected.
global_data.head()

,region,category,parameter,mode,powertrain,year,unit,value
0,Australia,Historical,EV stock share,Cars,EV,2011,percent,0.00039
1,Australia,Historical,EV sales share,Cars,EV,2011,percent,0.00650
2,Australia,Historical,EV sales,Cars,BEV,2011,Vehicles,49.00000
3,Australia,Historical,EV stock,Cars,BEV,2011,Vehicles,49.00000
4,Australia,Historical,EV stock,Cars,BEV,2012,Vehicles,220.00000


In [6]:
# Inspect the data types and missing values to understand the structure before cleaning.
global_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 12654 entries, 0 to 12653
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   region      12654 non-null  str    
 1   category    12654 non-null  str    
 2   parameter   12654 non-null  str    
 3   mode        12654 non-null  str    
 4   powertrain  12654 non-null  str    
 5   year        12654 non-null  int64  
 6   unit        12654 non-null  str    
 7   value       12654 non-null  float64
dtypes: float64(1), int64(1), str(6)
memory usage: 1.4 MB


In [7]:
# Review the numeric summary to spot unusual ranges or potential outliers.
global_data.describe()

,year,value
count,12654.000000,1.265400e+04
mean,2019.822112,4.273742e+05
std,5.476494,6.860498e+06
min,2010.000000,1.200000e-06
25%,2016.000000,2.000000e+00
50%,2020.000000,1.300000e+02
75%,2022.000000,5.500000e+03
max,2035.000000,4.400000e+08


# Data Cleaning and Exploration

There are a few things to note about the data after taking a quick glance.
1) There seems to be 6 non-numerical columns. I want to see what values those columns (`category`, `parameter`, `mode`, `powertrain`, and `unit`) can have.
2) Based upon the first few rows of the data, it appears that this data was formatted long where `value` describes the `parameter`. The way I envision the PowerBI dashboard to work, having wider data, instead of longer, would work better.
3) Since this data seems to only capture yearly statistics, a year-over-year (YoY) metric may be helpful to visualize.

## Step 1: Values

In [11]:
# Review the distinct values across the categorical fields used for filtering and grouping.
print(f'Category: {list(global_data["category"].unique())}')
print(f'Parameter: {list(global_data["parameter"].unique())}')
print(f'Mode: {list(global_data["mode"].unique())}')
print(f'Powertrain: {list(global_data["powertrain"].unique())}')
print(f'Unit: {list(global_data["unit"].unique())}')
print(f'Region: {list(global_data["region"].unique())}')

Category: ['Historical', 'Projection-STEPS', 'Projection-APS']
Parameter: ['EV stock share', 'EV sales share', 'EV sales', 'EV stock', 'EV charging points', 'Electricity demand', 'Oil displacement Mbd', 'Oil displacement, million lge']
Mode: ['Cars', 'EV', 'Buses', 'Vans', 'Trucks']
Powertrain: ['EV', 'BEV', 'PHEV', 'Publicly available fast', 'Publicly available slow', 'FCEV']
Unit: ['percent', 'Vehicles', 'charging points', 'GWh', 'Milion barrels per day', 'Oil displacement, million lge']
Region: ['Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'EU27', 'Europe', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea', 'Latvia', 'Lithuania', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland', 'Portugal', 'Rest of the world', 'Romania', 'Seychelles', 'Slovakia', 'Slov

A few definitions behind the values:
- `Oil displacement Mbd`: The amount of oil (gasoline and diesel) avoided by using EVs, in millions of barrels per day
- `Oil displacement, million lge`: The amount of fuel consumption avoided by using EVs, in million liters of gasoline equivalent

Since the focus of this project is historical sales of EVs, I'm going to filter out all other data.

In [36]:
# Filter to the historical, EV-focused rows needed for the dashboard.
# This removes non-EV categories, oil-related parameters, and future-year observations.
data_rel = global_data[(global_data['category'] == 'Historical')
                        & ~(global_data['region'].isin(['World','EU27','Europe',
                                                         'Rest of the world']))
                        # & (~global_data['parameter'].str.contains('Oil'))
                        & (global_data['powertrain'].str.contains('EV'))
                        & (global_data['year'] < datetime.today().year)
                        ]

In [37]:
# Pivot the long-form data so each metric becomes its own column for easier analysis.
pivoted = pd.pivot_table(data_rel,
                           index = ['year','region','powertrain','mode'],
                           columns = ['parameter'],
                           values = ['value'],
                           aggfunc = 'sum',
                           fill_value = 0)

In [38]:
# Simplify the multi-level column names produced by the pivot so they are easier to work with.
pivoted.columns = [i[1]
                   for i in pivoted.columns]

In [39]:
# Calculate year-over-year percent changes for each metric within each region/powertrain/mode grouping.
pivoted_yoy = pivoted.reset_index()\
                        .groupby(['region','powertrain','mode'])\
                        .apply(lambda x: x.set_index('year')\
                                           .pct_change())

In [40]:
# Rename the YoY columns to make the metric suffixes explicit in the final table.
pivoted_yoy.columns = ['_'.join([i,'yoy'])
                       for i in pivoted_yoy.columns]

In [41]:
# Combine the raw metric values with their year-over-year counterparts into one analysis table.
joined = pd.merge(pivoted,
                  pivoted_yoy,
                  how = 'outer',
                  left_index = True,
                  right_index = True)\
            .reset_index()\
            .sort_values(['region','powertrain','mode','year'])

In [42]:
# Preview the combined dataset to verify the reshaping and YoY merge look correct.
joined.head(10)

,year,region,powertrain,mode,EV sales,EV sales share,EV stock,EV stock share,Electricity demand,Oil displacement Mbd,"Oil displacement, million lge",EV sales_yoy,EV sales share_yoy,EV stock_yoy,EV stock share_yoy,Electricity demand_yoy,Oil displacement Mbd_yoy,"Oil displacement, million lge_yoy"
109,2011,Australia,BEV,Cars,49.0,0.0,49.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
248,2012,Australia,BEV,Cars,170.0,0.0,220.0,0.0,0.0,0.0,0.0,2.469388,NaN,3.489796,NaN,NaN,NaN,NaN
405,2013,Australia,BEV,Cars,190.0,0.0,410.0,0.0,0.0,0.0,0.0,0.117647,NaN,0.863636,NaN,NaN,NaN,NaN
580,2014,Australia,BEV,Cars,370.0,0.0,780.0,0.0,0.0,0.0,0.0,0.947368,NaN,0.902439,NaN,NaN,NaN,NaN
762,2015,Australia,BEV,Cars,760.0,0.0,1500.0,0.0,0.0,0.0,0.0,1.054054,NaN,0.923077,NaN,NaN,NaN,NaN
1002,2016,Australia,BEV,Cars,670.0,0.0,2200.0,0.0,0.0,0.0,0.0,-0.118421,NaN,0.466667,NaN,NaN,NaN,NaN
1270,2017,Australia,BEV,Cars,1200.0,0.0,3400.0,0.0,0.0,0.0,0.0,0.791045,NaN,0.545455,NaN,NaN,NaN,NaN
1559,2018,Australia,BEV,Cars,1800.0,0.0,5200.0,0.0,0.0,0.0,0.0,0.500000,NaN,0.529412,NaN,NaN,NaN,NaN
1852,2019,Australia,BEV,Cars,6300.0,0.0,12000.0,0.0,0.0,0.0,0.0,2.500000,NaN,1.307692,NaN,NaN,NaN,NaN
2176,2020,Australia,BEV,Cars,5200.0,0.0,17000.0,0.0,0.0,0.0,0.0,-0.174603,NaN,0.416667,NaN,NaN,NaN,NaN


In [44]:
# Export the joined dataset without geometry fields for simpler downstream use.
joined.to_csv('../Data/pivoted_data_yoy.csv',
              index = False)